In [ ]:
import json  # Thư viện để lưu và đọc file
import os   # Thư viện để kiểm tra file
from datetime import datetime  # Thư viện để làm việc với ngày tháng
import shutil  # Thư viện để sao lưu file

# Tên file lưu dữ liệu
DATA_FILE = "NHANVIEN.TXT"

# Hàm kiểm tra ngày hợp lệ (VD: 01/01/2000)
def is_valid_date(date_str):
    try:
        datetime.strptime(date_str, "%d/%m/%Y")  # Kiểm tra định dạng ngày
        return True
    except:
        return False

# Hàm nhập thông tin nhân viên
def input_employee():
    # Nhập họ tên
    while True:
        name = input("Nhập họ tên (tối đa 25 ký tự): ")
        if len(name) <= 25 and len(name) > 0:  # Kiểm tra tên hợp lệ
            break
        print("Tên không hợp lệ hoặc quá dài!")

    # Nhập ngày sinh
    while True:
        dob = input("Nhập ngày sinh (VD: 01/01/2000): ")
        if is_valid_date(dob):  # Kiểm tra ngày hợp lệ
            break
        print("Ngày sinh không đúng định dạng!")

    # Nhập ngày vào công ty
    while True:
        join_date = input("Nhập ngày vào công ty (VD: 01/01/2020): ")
        if is_valid_date(join_date):
            break
        print("Ngày vào không đúng định dạng!")

    # Nhập đơn vị
    department = input("Nhập đơn vị làm việc: ")

    # Nhập lương
    while True:
        try:
            salary = float(input("Nhập lương: "))
            if salary >= 0:  # Kiểm tra lương không âm
                break
            print("Lương phải lớn hơn hoặc bằng 0!")
        except:
            print("Lương phải là số!")

    # Tạo dictionary chứa thông tin nhân viên
    employee = {
        "name": name,
        "dob": dob,
        "join_date": join_date,
        "department": department,
        "salary": salary
    }
    return employee

# Hàm đọc dữ liệu từ file
def read_data():
    if not os.path.exists(DATA_FILE):  # Nếu file không tồn tại
        return []
    with open(DATA_FILE, 'r', encoding='utf-8') as file:
        try:
            data = json.load(file)  # Đọc dữ liệu JSON
            return data
        except:
            return []

# Hàm lưu dữ liệu vào file
def write_data(data):
    with open(DATA_FILE, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)  # Lưu dữ liệu JSON

# Hàm thêm nhân viên
def add_employee():
    employee = input_employee()  # Nhập thông tin
    data = read_data()  # Đọc danh sách hiện tại
    data.append(employee)  # Thêm nhân viên mới
    write_data(data)  # Lưu lại
    print("Đã thêm nhân viên!")

# Hàm in thông tin một nhân viên
def print_employee(employee):
    print(f"Họ tên: {employee['name']}")
    print(f"Ngày sinh: {employee['dob']}")
    print(f"Ngày vào: {employee['join_date']}")
    print(f"Đơn vị: {employee['department']}")
    print(f"Lương: {employee['salary']:,}")
    print("-" * 30)

# Hàm tìm kiếm nhân viên
def search_employee():
    name = input("Nhập tên cần tìm: ").lower()  # Chuyển thành chữ thường
    data = read_data()
    found = False
    for employee in data:
        if name in employee["name"].lower():  # Tìm tên khớp
            print_employee(employee)
            found = True
    if not found:
        print("Không tìm thấy nhân viên!")

# Hàm xóa nhân viên
def delete_employee():
    name = input("Nhập tên nhân viên cần xóa: ").lower()
    data = read_data()
    new_data = []
    found = False
    for employee in data:
        if name not in employee["name"].lower():  # Giữ lại nhân viên không khớp
            new_data.append(employee)
        else:
            found = True
    if found:
        confirm = input("Bạn có chắc muốn xóa? (y/n): ").lower()
        if confirm == 'y':
            write_data(new_data)  # Lưu danh sách mới
            print("Đã xóa nhân viên!")
        else:
            print("Đã hủy xóa!")
    else:
        print("Không tìm thấy nhân viên!")

# Hàm hiển thị tất cả nhân viên
def display_employees():
    data = read_data()
    if not data:
        print("Danh sách trống!")
        return
    for employee in data:
        print_employee(employee)
#--------------------------------------------------------------------------------------------------------------------
# Hàm lọc nhân viên theo lương
def filter_by_salary():
    try:
        min_salary = float(input("Nhập mức lương tối thiểu: "))
        data = read_data()
        found = False
        for employee in data:
            if employee["salary"] > min_salary:
                print_employee(employee)
                found = True
        if not found:
            print("Không tìm thấy nhân viên!")
    except:
        print("Lương phải là số!")

# Hàm sắp xếp theo lương
def sort_by_salary():
    data = read_data()
    choice = input("Sắp xếp tăng dần (1) hay giảm dần (2)? ")
    
    # Tạo danh sách lương và danh sách nhân viên tương ứng
    salaries = []
    for employee in data:
        salaries.append(employee["salary"])
    
    # Sắp xếp thủ công
    sorted_data = []
    if choice == '1':  # Tăng dần
        while salaries:
            min_salary = min(salaries)
            for employee in data:
                if employee["salary"] == min_salary and employee not in sorted_data:
                    sorted_data.append(employee)
                    salaries.remove(min_salary)
                    break
    elif choice == '2':  # Giảm dần
        while salaries:
            max_salary = max(salaries)
            for employee in data:
                if employee["salary"] == max_salary and employee not in sorted_data:
                    sorted_data.append(employee)
                    salaries.remove(max_salary)
                    break
    else:
        print("Lựa chọn không hợp lệ!")
        return
    
    for employee in sorted_data:
        print_employee(employee)

# Hàm phân tích lương
def analyze_salary():
    data = read_data()
    if not data:
        print("Danh sách trống!")
        return
    
    # Tính lương trung bình, cao nhất, thấp nhất
    total_salary = 0
    salaries = []
    for employee in data:
        total_salary += employee["salary"]
        salaries.append(employee["salary"])
    
    avg_salary = total_salary / len(data) if data else 0
    max_salary = max(salaries) if salaries else 0
    min_salary = min(salaries) if salaries else 0
    
    # Phân nhóm lương
    under_5m = 0
    from_5m_to_10m = 0
    above_10m = 0
    for salary in salaries:
        if salary < 5000000:
            under_5m += 1
        elif salary <= 10000000:
            from_5m_to_10m += 1
        else:
            above_10m += 1
    
    print(f"Lương trung bình: {avg_salary:,.0f}")
    print(f"Lương cao nhất: {max_salary:,.0f}")
    print(f"Lương thấp nhất: {min_salary:,.0f}")
    print("Phân nhóm lương:")
    print(f"Dưới 5 triệu: {under_5m} người")
    print(f"Từ 5-10 triệu: {from_5m_to_10m} người")
    print(f"Trên 10 triệu: {above_10m} người")

# Hàm lọc theo tuổi
def filter_by_age():
    data = read_data()
    if not data:
        print("Danh sách trống!")
        return
    
    today = datetime.now()
    under_30 = []
    from_30_to_40 = []
    above_40 = []
    
    for employee in data:
        dob = datetime.strptime(employee["dob"], "%d/%m/%Y")
        age = today.year - dob.year
        if (today.month, today.day) < (dob.month, dob.day):
            age -= 1  # Giảm tuổi nếu chưa đến sinh nhật
        
        if age < 30:
            under_30.append(employee)
        elif age <= 40:
            from_30_to_40.append(employee)
        else:
            above_40.append(employee)
    
    print("\nNhóm dưới 30 tuổi:")
    if under_30:
        for employee in under_30:
            print_employee(employee)
    else:
        print("Không có ai!")
    
    print("\nNhóm 30-40 tuổi:")
    if from_30_to_40:
        for employee in from_30_to_40:
            print_employee(employee)
    else:
        print("Không có ai!")
    
    print("\nNhóm trên 40 tuổi:")
    if above_40:
        for employee in above_40:
            print_employee(employee)
    else:
        print("Không có ai!")

# Hàm lọc theo đơn vị
def filter_by_department():
    dept = input("Nhập đơn vị cần tìm: ").lower()
    data = read_data()
    found = False
    for employee in data:
        if dept in employee["department"].lower():
            print_employee(employee)
            found = True
    if not found:
        print("Không tìm thấy nhân viên!")

# Hàm cập nhật nhân viên
def update_employee():
    name = input("Nhập tên nhân viên cần sửa: ").lower()
    data = read_data()
    for employee in data:
        if name in employee["name"].lower():
            print("Thông tin hiện tại:")
            print_employee(employee)
            print("Nhập thông tin mới (Enter để giữ nguyên):")
            
            new_name = input("Họ tên: ")
            if new_name and len(new_name) <= 25:
                employee["name"] = new_name
            
            new_dob = input("Ngày sinh (VD: 01/01/2000): ")
            if new_dob and is_valid_date(new_dob):
                employee["dob"] = new_dob
            
            new_join = input("Ngày vào (VD: 01/01/2020): ")
            if new_join and is_valid_date(new_join):
                employee["join_date"] = new_join
            
            new_dept = input("Đơn vị: ")
            if new_dept:
                employee["department"] = new_dept
            
            new_salary = input("Lương: ")
            if new_salary:
                try:
                    salary = float(new_salary)
                    if salary >= 0:
                        employee["salary"] = salary
                    else:
                        print("Lương không hợp lệ!")
                        return
                except:
                    print("Lương phải là số!")
                    return
            
            write_data(data)
            print("Đã cập nhật nhân viên!")
            return
    print("Không tìm thấy nhân viên!")

# Hàm thống kê theo đơn vị
def stats_by_department():
    data = read_data()
    if not data:
        print("Danh sách trống!")
        return
    
    departments = {}
    for employee in data:
        dept = employee["department"]
        if dept in departments:
            departments[dept] += 1
        else:
            departments[dept] = 1
    
    print("Số nhân viên theo đơn vị:")
    for dept, count in departments.items():
        print(f"{dept}: {count} người")

# Hàm sao lưu dữ liệu
def backup_data():
    if not os.path.exists(DATA_FILE):
        print("Không có dữ liệu để sao lưu!")
        return
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_file = f"NHANVIEN_{timestamp}.TXT"
    shutil.copy(DATA_FILE, backup_file)
    print(f"Đã sao lưu vào: {backup_file}")

# Hàm kiểm tra nhân viên sắp nghỉ hưu (từ 55 tuổi)
def check_retirement():
    data = read_data()
    if not data:
        print("Danh sách trống!")
        return
    
    today = datetime.now()
    found = False
    for employee in data:
        dob = datetime.strptime(employee["dob"], "%d/%m/%Y")
        age = today.year - dob.year
        if (today.month, today.day) < (dob.month, dob.day):
            age -= 1
        if age >= 55:
            print_employee(employee)
            found = True
    if not found:
        print("Không có nhân viên từ 55 tuổi trở lên!")

# Hàm chính hiển thị menu
def main():
    while True:
        print("\n=== QUẢN LÝ NHÂN VIÊN ===")
        print("1. Thêm nhân viên")
        print("2. Tìm nhân viên")
        print("3. Xóa nhân viên")
        print("4. Hiển thị tất cả")
        print("5. Lọc theo lương")
        print("6. Sắp xếp theo lương")
        print("7. Phân tích lương")
        print("8. Lọc theo tuổi")
        print("9. Kiểm tra nhân viên sắp nghỉ hưu")
        print("10. Lọc theo đơn vị")
        print("11. Sửa thông tin nhân viên")
        print("12. Thống kê theo đơn vị")
        print("13. Sao lưu dữ liệu")
        print("0. Thoát")
        
        choice = input("Chọn số từ 0-13: ")
        
        if choice == '0':
            print("Tạm biệt!")
            break
        elif choice == '1':
            add_employee()
        elif choice == '2':
            search_employee()
        elif choice == '3':
            delete_employee()
        elif choice == '4':
            display_employees()
        elif choice == '5':
            filter_by_salary()
        elif choice == '6':
            sort_by_salary()
        elif choice == '7':
            analyze_salary()
        elif choice == '8':
            filter_by_age()
        elif choice == '9':
            check_retirement()
        elif choice == '10':
            filter_by_department()
        elif choice == '11':
            update_employee()
        elif choice == '12':
            stats_by_department()
        elif choice == '13':
            backup_data()
        else:
            print("Số không hợp lệ, thử lại!")

# Chạy chương trình
if __name__ == "__main__":
    main()


=== QUẢN LÝ NHÂN VIÊN ===
1. Thêm nhân viên
2. Tìm nhân viên
3. Xóa nhân viên
4. Hiển thị tất cả
5. Lọc theo lương
6. Sắp xếp theo lương
7. Phân tích lương
8. Lọc theo tuổi
9. Kiểm tra nhân viên sắp nghỉ hưu
10. Lọc theo đơn vị
11. Sửa thông tin nhân viên
12. Thống kê theo đơn vị
13. Sao lưu dữ liệu
0. Thoát
